In [ ]:
import pandas as pd
import requests
import os
from dotenv import load_dotenv



load_dotenv(override=True)
api_key = os.getenv("api_key")

def get_release_year(imdbID):
    """Obtem a data de lançamento do filme por meio da consulta à API do IMDB."""
    try:
        url = f"http://www.omdbapi.com/?i={imdbID}&apikey={api_key}"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        return data.get("Released")
    except Exception as e:
        print(f"Erro ao buscar dados da API:{e}")
        raise


def get_data():
    """Faz a extração dos dados do arquivo .csv e transforma em um dataframe do Pandas."""
    path = r"C:\Projetos python\Limpeza_dados-Pandas\Messy_data\messy_IMDB_dataset.csv"
    df = pd.read_csv(path,sep=';',encoding='latin-1')
    return df


In [ ]:
# Exploração dos dados:

df = get_data()

df.info()

df.describe()

print(df.nunique())

df.dropna(how='all')

df = df.drop('Unnamed: 8',axis = 1)

df.drop_duplicates()

display(df)

In [ ]:

# Tratamento de coluna do tempo de duração do filme:

df = df.copy()

df["Duration"]= df["Duration"].astype(str).str.strip()
df["Duration"] = df["Duration"].str.replace(r'[^0-9]','',regex=True)

df["Duration"].unique()

df[df["Duration"] == ""]
df["Duration"] = df["Duration"].replace('', pd.NA)
df["Duration"] = pd.to_numeric(df["Duration"], errors='coerce')



df["Duration"] = df["Duration"].fillna(
    df.groupby("Genrë¨")["Duration"].transform("median")
)



In [ ]:

# Tratamento da coluna de paises do filme

paises = {'USA':'United States', 'US.':'United States', 'US': 'United States','Italy1':'Italy',
          'UK':'United Kingdom','New Zesland':'New Zeland','New Zealand':'New Zeland'}

for i,j in paises.items():
    df["Country"] = df["Country"].str.replace(i,j)

df.sample(20)



In [ ]:

# Tratamento da coluna de data de lancamento do filme.

df["Release year"] = df["Release year"].astype(str).str.strip().str.lower()
df["Release year"] = df["Release year"].str.replace(r'\s+', ' ', regex=True)

df["Release year"] = df["Release year"].str.replace(r'(\d+)(st|nd|rd|th)', r'\1', regex=True)
df["Release year"] = df["Release year"].str.replace(r'\b(of|year|the)\b', '', regex=True)
df["Release year"] = df["Release year"].str.replace(r'\s+', ' ', regex=True).str.strip()

meses = {
    'january': '01', 'february': '02', 'march': '03', 'april': '04',
    'may': '05', 'june': '06', 'july': '07', 'august': '08',
    'september': '09', 'october': '10', 'november': '11', 'december': '12',
    'enero': '01', 'febrero': '02', 'marzo': '03', 'abril': '04',
    'mayo': '05', 'junio': '06', 'julio': '07', 'agosto': '08',
    'septiembre': '09', 'octubre': '10', 'noviembre': '11', 'diciembre': '12'
}

for nome, numero in meses.items():
    df["Release year"] = df["Release year"].str.replace(nome, numero, regex=False)

df["Release year"] = df["Release year"].str.replace(r'[/\s]', '-', regex=True)
df["Release year"] = df["Release year"].str.replace(r'-+', '-', regex=True).str.strip('-')

meses_abrev = {
    'jan': '01', 'feb': '02', 'mar': '03', 'apr': '04',
    'may': '05', 'jun': '06', 'jul': '07', 'aug': '08',
    'sep': '09', 'oct': '10', 'nov': '11', 'dec': '12'
}

for nome, numero in meses_abrev.items():
    df["Release year"] = df["Release year"].str.replace(nome, numero, regex=False)


df["Release year"] = df["Release year"].str.replace(',', '', regex=False)

teste = pd.to_datetime(df["Release year"], errors='coerce')


tentativa2 = pd.to_datetime(df["Release year"], errors='coerce', dayfirst=True)
df["Release year"] = teste.combine_first(tentativa2)









In [ ]:

# Limpando coluna de receita

df["Income"] = df["Income"].str.replace(r'$|[^0-9]','',regex=True)

df.loc[df["Income"].isnull()]

df = df.dropna(how='all')

print(df["Income"].unique())

df["Income"] = pd.to_numeric(df["Income"],errors='coerce')
df.sample(10)

In [ ]:
df = df.drop(columns=["Content Rating"])


In [ ]:
# # Tratamento da coluna de score e renomeando colunas.

df = df.rename(columns={"IMBD title ID":"ID_movie","Original titlÊ":"original_title","Release year":"release_year",
"Genrë¨":"genre"," Votes ":"votes","Duration":"duration","Income":"income","Score":"score","Country":"country","Director":"director"})

df["score"] = df["score"].str.extract(r"(\d+(?:\.\d+)?)")

df["score"] = pd.to_numeric(df["score"])

df["score"].value_counts()



In [ ]:
# Tratando coluna de votos

df["votes"] = df["votes"].str.replace('.','').str.strip()
df["votes"].unique()
df["votes"] = pd.to_numeric(df["votes"])

In [ ]:
# Substituindo os nulos com dados da API do IMDB.

mask_missing = df["release_year"].isna()

df.loc[mask_missing, "release_year"] = pd.to_datetime(

    df.loc[mask_missing, "ID_movie"].apply(get_release_year)
    
    )

df.dtypes

# df.isnull().sum()


In [ ]:
#Carregando arquivo limpo e tratado.


df.to_csv('Cleaned_data/imbd_dataset_cleaned.csv',index=False)